<a href="https://colab.research.google.com/github/Shwoouu/AAI2025/blob/dev/Coding_Exercise_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Part 2: Customer Churn Prediction using Logistic Regression

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load the same dataset
df = pd.read_csv("/content/paris_housing.csv")

# Use 100 records
df = df.head(100).copy()

# Create a customer churn variable
# Customers with fewer rooms and fewer previous owners are
# given a higher chance of being classified as churned.
df["churn"] = (
    (df["numberOfRooms"] < df["numberOfRooms"].median()) &
    (df["numPrevOwners"] < df["numPrevOwners"].median())
).astype(int)

# Customer features
X = df[
    ["numberOfRooms", "price", "numPrevOwners", "cityPartRange", "cityCode"]
]

y = df["churn"]

# Numerical and categorical features
numeric_features = [
    "numberOfRooms",
    "price",
    "numPrevOwners",
    "cityPartRange"
]

categorical_features = ["cityCode"]

# Scale numerical data and encode city
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("location", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Create Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(max_iter=1000))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train the model
model.fit(X_train, y_train)

# Predict churn probabilities
probabilities = model.predict_proba(X_test)[:, 1]

# Use 0.5 as the churn threshold
predictions = (probabilities >= 0.5).astype(int)

# Model performance
print("===== MODEL PERFORMANCE =====")
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Predict a new customer
new_customer = pd.DataFrame({
    "numberOfRooms": [5],
    "price": [5000000],
    "numPrevOwners": [2],
    "cityPartRange": [5],
    "cityCode": [df["cityCode"].iloc[0]]
})

churn_probability = model.predict_proba(new_customer)[0][1]
churn_prediction = int(churn_probability >= 0.5)

print("\n===== NEW CUSTOMER PREDICTION =====")
print("Churn probability:", round(churn_probability, 3))
print("Churn percentage:", round(churn_probability * 100, 1), "%")

if churn_prediction == 1:
    print("Classification: AT RISK OF CHURNING")
else:
    print("Classification: NOT AT RISK OF CHURNING")

# Display model coefficients
features = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["logistic_regression"].coef_[0]

print("\n===== MODEL COEFFICIENTS =====")

for feature, coefficient in zip(features, coefficients):
    print(f"{feature}: {coefficient:.3f}")

===== MODEL PERFORMANCE =====
Accuracy: 0.95

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.83      1.00      0.91         5

    accuracy                           0.95        20
   macro avg       0.92      0.97      0.94        20
weighted avg       0.96      0.95      0.95        20


===== NEW CUSTOMER PREDICTION =====
Churn probability: 0.947
Churn percentage: 94.7 %
Classification: AT RISK OF CHURNING

===== MODEL COEFFICIENTS =====
numeric__numberOfRooms: -1.985
numeric__price: -0.645
numeric__numPrevOwners: -1.801
numeric__cityPartRange: 0.275
location__cityCode_339: 0.397
location__cityCode_1690: -0.006
location__cityCode_2922: -0.008
location__cityCode_3406: 0.527
location__cityCode_4863: -0.003
location__cityCode_5484: -0.128
location__cityCode_5898: -0.109
location__cityCode_6450: -0.178
location__cityCode_6517: -0.004
location__cityCode_6739: -0.004
location__cityCod